c:\Users\delga\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


❌ Error: No se encuentra la carpeta 'data'. Asegúrate de que existe.
🐢 Procesando con: CPU (Va a tardar un poco, ten paciencia)
⏳ Cargando reviews...
⚠️ No se encontró la columna 'date', cargando sin orden temporal...


FileNotFoundError: [Errno 2] No such file or directory: 'data\\reviews.csv'

In [2]:
# ==============================================================================
# 1. IMPORTACIONES Y CONFIGURACIÓN DEL ENTORNO DE DEEP LEARNING
# ==============================================================================
from transformers import pipeline
from tqdm.auto import tqdm
import warnings
import pandas as pd
import numpy as np
import re
import os
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocesamiento (Scikit-Learn)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Deep Learning (TensorFlow & Keras)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Configuración visual
sns.set_theme(style="whitegrid")
print(f"✅ TensorFlow Version: {tf.__version__}")

✅ TensorFlow Version: 2.21.0


In [ ]:
# ==============================================================================
# 1. CONFIGURACIÓN
# ==============================================================================
warnings.filterwarnings('ignore')
tqdm.pandas() # Activa la barra de progreso de Pandas

# Configuramos la ruta local a tu carpeta de datos
PATH_DATA = 'data'
if not os.path.exists(PATH_DATA):
    print(f"❌ Error: No se encuentra la carpeta '{PATH_DATA}'. Asegúrate de que existe.")

# Verificamos si hay GPU disponible en tu equipo local
if torch.cuda.is_available():
    device = 0
    print("🚀 Procesando con: GPU (NVIDIA)")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 0
    print("🚀 Procesando con: GPU (Mac Apple Silicon)")
else:
    device = -1
    print("🐢 Procesando con: CPU (Va a tardar un poco, ten paciencia)")

# ==============================================================================
# 2. CARGA Y MUESTREO ESTRATIFICADO (TOP 10 POR PISO)
# ==============================================================================
print("⏳ Cargando reviews...")
ruta_reviews = os.path.join(PATH_DATA, 'reviews.csv')

# Cargamos solo las columnas necesarias para no saturar la RAM
try:
    df_reviews = pd.read_csv(ruta_reviews, usecols=['listing_id', 'comments', 'date']).dropna()
    df_reviews = df_reviews.sort_values(by=['listing_id', 'date'], ascending=[True, False])
except Exception:
    print("⚠️ No se encontró la columna 'date', cargando sin orden temporal...")
    df_reviews = pd.read_csv(ruta_reviews, usecols=['listing_id', 'comments']).dropna()

print("📊 Aplicando muestreo (Top 10 reseñas por alojamiento)...")
df_reviews = df_reviews.groupby('listing_id').head(10).copy()

def limpiar_texto(texto):
    texto = str(texto)
    texto = re.sub(r'<br\s*/?>|[\r\n]+', ' ', texto)
    return re.sub(r'\s+', ' ', texto).strip()

print("🧹 Limpiando texto...")
df_reviews['comments'] = df_reviews['comments'].apply(limpiar_texto)
# Filtramos comentarios que se queden muy cortos tras limpiar
df_reviews = df_reviews[df_reviews['comments'].str.len() > 10].copy()

print(f"✅ Listas para procesar: {len(df_reviews)} reseñas.")

# ==============================================================================
# 3. INFERENCIA CON TRANSFORMER MULTILINGÜE
# ==============================================================================
print("🤖 Iniciando análisis de sentimiento con Deep Learning...")

sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    truncation=True,
    max_length=512,
    device=device
)

def obtener_score(texto):
    try:
        resultado = sentiment_analyzer(texto[:512])[0]
        # Transforma de '1-5 stars' a un score de 0.0 a 1.0
        return (int(resultado['label'].split()[0]) - 1) / 4.0
    except:
        return 0.5 # Nota neutra si hay error

# progress_apply muestra una barra de carga para que sepas por dónde va
df_reviews['score_sentimiento'] = df_reviews['comments'].progress_apply(obtener_score)

# ==============================================================================
# 4. AGRUPACIÓN Y GUARDADO FINAL
# ==============================================================================
print("📊 Agrupando resultados por alojamiento...")
df_nlp_final = df_reviews.groupby('listing_id')['score_sentimiento'].mean().reset_index()

ruta_salida = os.path.join(PATH_DATA, 'reviews_scores_nlp.csv')
df_nlp_final.to_csv(ruta_salida, index=False)

print(f"🏁 ¡PROCESO COMPLETADO! El archivo final está guardado en: {ruta_salida}")

In [ ]:
# ==============================================================================
# 2. CARGA DE DATOS Y LIMPIEZA INICIAL
# ==============================================================================
PATH_DATA = 'data'
ruta_dataset = os.path.join(PATH_DATA, 'dataset_ia_final.csv')

print("⏳ Cargando dataset maestro...")
df = pd.read_csv(ruta_dataset, low_memory=False)

# 2.1 Definir las variables que usará el modelo (Features) y lo que queremos predecir (Target)
features_num = ['accommodates', 'bedrooms', 'price', 'number_of_reviews', 'review_scores_rating', 'renta_media', 'score_sentimiento']
features_cat = ['property_type', 'room_type', 'host_is_superhost']
target = 'tasa_ocupacion'

# Nos quedamos solo con lo que importa
df_model = df[features_num + features_cat + [target]].copy()

# 2.2 Tratamiento de Nulos (Crucial para Keras)
print("🧹 Tratando valores nulos para la Red Neuronal...")

# Si un piso no tiene reseñas, le damos valores neutros/medianos
df_model['score_sentimiento'] = df_model['score_sentimiento'].fillna(0.5)
df_model['review_scores_rating'] = df_model['review_scores_rating'].fillna(df_model['review_scores_rating'].median())
df_model['number_of_reviews'] = df_model['number_of_reviews'].fillna(0)

# Si un piso cayó fuera del mapa y no tiene renta media, le damos la mediana de Sevilla
df_model['renta_media'] = df_model['renta_media'].fillna(df_model['renta_media'].median())

# Si no tiene dato de habitaciones, asumimos 1 por defecto
df_model['bedrooms'] = df_model['bedrooms'].fillna(1.0)

# Tratamiento de booleanos (Superhost a veces viene como 't'/'f')
df_model['host_is_superhost'] = df_model['host_is_superhost'].astype(str).replace({'t': 'True', 'f': 'False', 'nan': 'False'})

# ELIMINAR filas donde no sepamos la ocupación (No podemos enseñar a la IA con exámenes sin nota)
df_model = df_model.dropna(subset=[target])

print(f"✅ Dataset listo para transformación: {df_model.shape[0]} alojamientos y {df_model.shape[1]} variables.")